# CAP F1 Abstention
This notebook (hackily) adapts Jimin's CAP F1 code to use with our self-consistency based absention work.

## Include Library

In [1]:
from datetime import datetime
import os
import random

# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    FEWSHOT_DEDUP_MESSAGES,
    FEWSHOT_RECALL_MESSAGES,
    FEWSHOT_PRECISION_MESSAGES,
)

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

## 1. build API + processor (inject few-shot examples if you want)
currently fewshot example is at fewshot_examples.py

In [2]:
# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=FEWSHOT_DEDUP_MESSAGES,  # or None
    fewshot_recall=FEWSHOT_RECALL_MESSAGES,  # or None
    fewshot_precision=FEWSHOT_PRECISION_MESSAGES,  # or None
)

Initialized LLM client with gpt-4.1-2025-04-14 and temperature = 0.2


## 2. Load Data

In [3]:
def get_random_sample(dataset, count=1):
    """Get a random sample from the dataset

    Args:
        dataset (list): List of dictionaries
        count (int, optional): Number of samples to return. Defaults to 1.

    Returns:
        list: List of dictionaries (random samples)
    """
    return random.sample(dataset, count)

In [4]:
print("Loading abstention dataset...")

# number of data points testing
LIMIT = 50

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

# create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

# features that we need to extract from the original dataset
org_caption_dataset = ResultsRepo.read_json(
    "test-data-scored-with-mt-metrics_300-examples_2025-08-12_12-29-55.json"
)
# org_caption_dataset = org_caption_dataset[:LIMIT]
org_caption_dataset = get_random_sample(org_caption_dataset, count=LIMIT)
org_caption_dataset

Loading abstention dataset...


[{'image_id': 5583,
  'file_name': 'VizWiz_train_00005583.jpg',
  'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00005583.jpg',
  'human_captions': 'A can of canned green beans on the counter\nSide of a can of green beans showing partial nutritional information.\nA container of vegetables sits on a flat surface.\nA can of green beans sitting on a black surface.\nTin can with a green and white label and pictures of green beans.',
  'annotator': 'Anne Marie',
  'annotation': 'can green beans',
  'gpt4o_caption': 'A can with an image of green beans on its label, placed on a brown surface.',
  'gpt4o_code': 'yes',
  'greedy_response': 'A can with an image of green beans on its label, placed on a carpeted surface.',
  'additional_responses': ['A can with a label showing images of green beans, with a mostly green background. The can is sitting on a brown textured surface.',
   'A can with an image of green beans on its label, placed on a carpeted surface.

### Hacks for getting abstention data to work with current CAP F1

1. Change human_captions to human_captions_crowdworkers
2. Create a new human_captions field with the following:

```python
'human_captions': [
    {
        'caption': <greedy caption goes here>,
        'is_precanned': False,
        'is_rejected': False
    }
]
```
3. Create a model_captions to hold samples as:
```python
'model_captions': [
    {
        'model_name': 'sample_1',
        'caption': ''
    },
    {
        'model_name': 'sample_2',
        'caption': ''
    },
    ...
    {
        'model_name': 'sample_10',
        'caption': ''
    },
],
```

In [6]:
for item in org_caption_dataset:
    # hack 1
    item["crowdworker_captions"] = item["human_captions"]
    del item["human_captions"]

    # hack 2
    item["human_captions"] = [
        {
            "caption": item["greedy_response"],
            "is_precanned": False,
            "is_rejected": False,
        }
    ]

    # hack 3
    item["model_captions"] = [
        {
            "model_name": f"sample_{index + 1}",
            "caption": sample,
        }
        for index, sample in enumerate(item["additional_responses"])
    ]

    # add an evaluation field
    item["evaluation"] = {}
org_caption_dataset[0]

{'image_id': 5583,
 'file_name': 'VizWiz_train_00005583.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00005583.jpg',
 'annotator': 'Anne Marie',
 'annotation': 'can green beans',
 'gpt4o_caption': 'A can with an image of green beans on its label, placed on a brown surface.',
 'gpt4o_code': 'yes',
 'greedy_response': 'A can with an image of green beans on its label, placed on a carpeted surface.',
 'additional_responses': ['A can with a label showing images of green beans, with a mostly green background. The can is sitting on a brown textured surface.',
  'A can with an image of green beans on its label, placed on a carpeted surface.',
  'A can with an image of green beans on the label, set against a green-colored background. The can is placed on a carpeted surface.',
  'A can with a label featuring green beans. The background and part of the label are green with no visible brand name.',
  'A metal can with an image of green beans on its labe

### 3. generate atomics

In [7]:
# 3) generate atomics
print(f"Generating atomic statements using {llm.model}")
T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(
    org_caption_dataset, limit=LIMIT
)

# 3.1) save intermediate
print("Saving intermediate results...")
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset,
    T_atomics=T_atomics,
    g_atomics=g_atomics,
    parsed_T=parsed_T,
    T_org=all_human_captions,
    limit=LIMIT,
)

Generating atomic statements using gpt-4.1-2025-04-14


100%|██████████| 50/50 [15:12<00:00, 18.26s/it]

Saving intermediate results...
Saved JSON to: results/2025-08-20_23-23/intermediate_2025-08-20_23-23.json


### 4. evaluate and get recall and precision
- match human caption to model caption
- create recall and precision data

In [8]:
# before calculating F1 score, match sentences between human generated and model generated
print("Evaluating atomic statements...")
eval_out = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

# 4.1) save evaluation results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=eval_out,
    limit=LIMIT,
)

Evaluating atomic statements...


  2%|▏         | 1/50 [00:55<44:57, 55.06s/it]

[sample_5] Recall mismatch: len T=4 vs TP+FN=4


 12%|█▏        | 6/50 [06:20<50:16, 68.57s/it]

[sample_7] Recall mismatch: len T=4 vs TP+FN=4


...

[sample_3] Precision mismatch: len G=6 vs TP+FP=6


 48%|████▊     | 24/50 [30:22<36:07, 83.38s/it]

[sample_10] Precision mismatch: len G=6 vs TP+FP=6


100%|██████████| 50/50 [57:44<00:00, 69.30s/it]

Saved JSON to: results/2025-08-20_23-23/eval_2025-08-20_23-23.json


### 5. calculate cap f1 score


In [9]:
# 5) calculate cap f1 score
cap_scores = proc.calculate_cap_f1(eval_out)

# 5.1) save cap f1 score results
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=cap_scores,
    limit=LIMIT,
)

100%|██████████| 50/50 [00:00<00:00, 146244.91it/s]


Saved JSON to: results/2025-08-20_23-23/final_2025-08-20_23-23.json


In [11]:
# 6) Final JSON → CSV
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

Saving final results into csv...
CSV file saved to: results/2025-08-20_23-23/final_2025-08-20_23-23.csv
